 # Lineups
 
 
 ## API calls
 
 
 ### Eligible Players 
 Pull a list of players who have a game scheduled on a given date

In [2]:
import requests
import json

games_url = "https://tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com/getMLBGamesForDate"

querystring = {"gameDate":"20250415"}

headers = {
	"x-rapidapi-key": "f37166c3b1msh65b77f8591cec0cp158d02jsnf64afaedba40",
	"x-rapidapi-host": "tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com"
}

response1 = requests.get(games_url, headers=headers, params=querystring).json()

games = response1['body']

print(games)

with open('games.json', 'w') as f:
    json.dump(games, f, indent=2)

[{'gameID': '20250415_OAK@CHW', 'gameType': 'REGULAR_SEASON', 'away': 'OAK', 'gameTime': '7:40p', 'gameDate': '20250415', 'teamIDHome': '6', 'gameTime_epoch': '1744760400.0', 'probableStartingLineups': {'away': [], 'home': []}, 'teamIDAway': '20', 'probableStartingPitchers': {'away': '', 'home': ''}, 'home': 'CHW'}, {'gameID': '20250415_ARI@MIA', 'gameType': 'REGULAR_SEASON', 'away': 'ARI', 'gameTime': '6:40p', 'gameDate': '20250415', 'teamIDHome': '15', 'gameTime_epoch': '1744756800.0', 'probableStartingLineups': {'away': [], 'home': []}, 'teamIDAway': '1', 'probableStartingPitchers': {'away': '', 'home': ''}, 'home': 'MIA'}, {'gameID': '20250415_ATL@TOR', 'gameType': 'REGULAR_SEASON', 'away': 'ATL', 'gameTime': '7:07p', 'gameDate': '20250415', 'teamIDHome': '29', 'gameTime_epoch': '1744758420.0', 'probableStartingLineups': {'away': [], 'home': []}, 'teamIDAway': '2', 'probableStartingPitchers': {'away': '', 'home': ''}, 'home': 'TOR'}, {'gameID': '20250415_CHC@SD', 'gameType': 'REGUL

In [3]:
players_url = "https://tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com/getMLBPlayerList"

headers = {
	"x-rapidapi-key": "f37166c3b1msh65b77f8591cec0cp158d02jsnf64afaedba40",
	"x-rapidapi-host": "tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com"
}

response2 = requests.get(players_url, headers=headers).json()

players = response2['body']


with open('players.json', 'w') as f:
    json.dump(players, f, indent=2)

In [4]:
teams_in_games = set()
for g in games:
    teams_in_games.add(g['home'])
    teams_in_games.add(g['away'])

# 2) Filter the players:
#    - Must have a team in teams_in_games
#    - Must NOT have a position of 'P'
filtered_players = [
    p for p in players
    if p['team'] in teams_in_games and p['pos'] != 'P'
]

# 3) Create your final filtered dictionary, if desired
eligible_players = {'players': filtered_players}

# Now `filtered_data` is a dictionary containing only players who
# have a `team` value that appears as a home or away team in your `games`.
print(eligible_players)

{'players': [{'pos': 'LF', 'playerID': '694362', 'team': 'CIN', 'longName': 'Blake Dunn', 'teamID': '7'}, {'pos': 'SS', 'playerID': '669397', 'team': 'ATL', 'longName': 'Nick Allen', 'teamID': '2'}, {'pos': 'OF', 'playerID': '650559', 'team': 'ATL', 'longName': 'Bryan De La Cruz', 'teamID': '2'}, {'pos': '3B', 'playerID': '805367', 'team': 'CHW', 'longName': 'Chase Meidroth', 'teamID': '6'}, {'pos': '2B', 'playerID': '670032', 'team': 'LAA', 'longName': 'Nicky Lopez', 'teamID': '13'}, {'pos': '2B', 'playerID': '666158', 'team': 'CIN', 'longName': 'Gavin Lux', 'teamID': '7'}, {'pos': 'SS', 'playerID': '624641', 'team': 'PHI', 'longName': 'Edmundo Sosa', 'teamID': '21'}, {'pos': 'SS', 'playerID': '621028', 'team': 'LAA', 'longName': 'Kevin Newman', 'teamID': '13'}, {'pos': '1B', 'playerID': '665019', 'team': 'PHI', 'longName': 'Kody Clemens', 'teamID': '21'}, {'pos': '2B', 'playerID': '642708', 'team': 'WAS', 'longName': 'Amed Rosario', 'teamID': '30'}, {'pos': 'RF', 'playerID': '657656'

In [9]:
import pandas as pd

df = pd.DataFrame(eligible_players)
df = df["players"].apply(pd.Series)
df.head()
df.pos.value_counts()



OF    178
C      78
SS     68
3B     63
2B     56
1B     51
DH      5
Name: pos, dtype: int64

In [6]:
#clean data so all CF, LF, or RF are listed as OF
for player in eligible_players["players"]:
    if player["pos"] in ["LF", "RF", "CF"]:
        player["pos"] = "OF"
    elif player["pos"] == "IF":
        player["pos"] = "2B"
    elif player["pos"] == "TWP":
        player["pos"] = "DH"

In [8]:
#save file of eligible players to draft
with open('eligible_players.json', 'w') as f:
    json.dump(eligible_players, f, indent=2)

### Play-by-Play data

In [34]:
import requests

url = "https://tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com/getMLBBoxScore"

querystring = {"gameID":"20250401_BAL@COL_pmaloney046","playerStatsFormat":"list","startingLineups":"true","fantasyPoints":"true","battingR":"1","battingTB":"1","battingRBI":"1","battingBB":"1","battingSO":"-1","baseRunningSB":"1","pitchingIP":"3","pitchingH":"-1","pitchingER":"-1","pitchingBB":"-1","pitchingSO":"1","pitchingW":"2","pitchingL":"-2","pitchingHold":"2","pitchingSave":"2"}

headers = {
	"x-rapidapi-key": "f37166c3b1msh65b77f8591cec0cp158d02jsnf64afaedba40",
	"x-rapidapi-host": "tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

{'statusCode': 200, 'body': {'GameLength': '2:22', 'Umpires': 'HP: Bill Miller. 1B: Malachi Moore. 2B: Tripp Gibson. 3B: Emil Jimenez.', 'gameStatus': 'Completed', 'Attendance': '32,961', 'teamStats': {'away': {'Pitching': {'Groundouts': '9', 'Balk': '0', 'Wild Pitch': '1', 'Flyouts': '1', 'Inherited Runners': '0', 'Batters Faced': '34', 'Pitches': '118', 'Strikes': '83', 'Inherited Runners Scored': '0'}, 'BaseRunning': {'CS': '0', 'SB': '2', 'PO': '0'}, 'Fielding': {'E': '0', 'Pickoffs': '0', 'Passed Ball': '0'}, 'Hitting': {'BB': '4', 'AB': '35', 'H': '9', 'IBB': '0', 'HR': '1', 'TB': '13', '3B': '0', 'GIDP': '1', '2B': '1', 'R': '6', 'AVG': '0.257', 'SF': '0', 'SAC': '0', 'HBP': '1', 'RBI': '5', 'SO': '6'}}, 'home': {'Pitching': {'Groundouts': '11', 'Balk': '0', 'Wild Pitch': '0', 'Flyouts': '6', 'Inherited Runners': '2', 'Batters Faced': '40', 'Pitches': '144', 'Strikes': '88', 'Inherited Runners Scored': '0'}, 'BaseRunning': {'CS': '0', 'SB': '0', 'PO': '0'}, 'Fielding': {'E': '2'

In [36]:
import json
import time
import requests

def get_all_play_by_play(game_id, headers):
    """
    Makes an API call for a single game_id and returns the 'allPlayByPlay' data if available.
    """
    url = "https://tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com/getMLBBoxScore"
    game_id_suffix =  f"{game_id}"
    querystring = {
        "gameID": game_id
    }
    
    headers = {
        "x-rapidapi-key": "f37166c3b1msh65b77f8591cec0cp158d02jsnf64afaedba40",
        "x-rapidapi-host": "tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)
    
    if response.status_code != 200:
        print(f"[ERROR] Request for {game_id} failed with status {response.status_code}.")
        return None
    
    data = response.json()
    
    # The 'allPlayByPlay' data typically lives at data["body"]["allPlayByPlay"].
    if "body" in data and "allPlayByPlay" in data["body"]:
        return data["body"]["allPlayByPlay"]
    else:
        print(f"[WARN] 'allPlayByPlay' key not found for {game_id}.")
        return None

def main():
    # 1) Load the list of games from a local JSON file
    with open("games.json", "r") as f:
        games = json.load(f)

    # 2) Prepare the RapidAPI request headers
    headers = {
        "x-rapidapi-key": "YOUR_RAPIDAPI_KEY",
        "x-rapidapi-host": "tank01-mlb-live-in-game-real-time-statistics.p.rapidapi.com"
    }

    # 3) Loop through each game, fetching 'allPlayByPlay'
    results = []
    for game in games:
        game_id = game["gameID"]
        print(f"Fetching 'allPlayByPlay' for {game_id}...")
        
        play_by_play_data = get_all_play_by_play(game_id, headers)
        results.append({
            "gameID": game_id,
            "allPlayByPlay": play_by_play_data
        })
        
        # Sleep to avoid hitting rate limits (optional)
        time.sleep(1)

    # 4) Save all 'allPlayByPlay' data to a single JSON file
    with open("allPlayByPlay_data.json", "w") as outfile:
        json.dump(results, outfile, indent=2)

    print("All 'allPlayByPlay' data has been saved to 'allPlayByPlay_data.json'.")

if __name__ == "__main__":
    main()


Fetching 'allPlayByPlay' for 20250401_DET@SEA...
Fetching 'allPlayByPlay' for 20250401_KC@MIL...
Fetching 'allPlayByPlay' for 20250401_ARI@NYY...
Fetching 'allPlayByPlay' for 20250401_LAA@STL...
Fetching 'allPlayByPlay' for 20250401_PIT@TB...
Fetching 'allPlayByPlay' for 20250401_NYM@MIA...
Fetching 'allPlayByPlay' for 20250401_SF@HOU...
Fetching 'allPlayByPlay' for 20250401_CHC@OAK...
Fetching 'allPlayByPlay' for 20250401_ATL@LAD...
Fetching 'allPlayByPlay' for 20250401_TEX@CIN...
Fetching 'allPlayByPlay' for 20250401_CLE@SD...
Fetching 'allPlayByPlay' for 20250401_MIN@CHW...
Fetching 'allPlayByPlay' for 20250401_WAS@TOR...
All 'allPlayByPlay' data has been saved to 'allPlayByPlay_data.json'.


In [40]:
import json
from collections import Counter

# Open the JSON file (adjust the path if needed)
with open("allPlayByPlay_data.json", "r") as f:
    data = json.load(f)

# Initialize a list to hold all the results
results_list = []

# Loop through each game and each play to extract the result
for game in data:
    # Use .get() to safely access "allPlayByPlay" in case it's missing
    plays = game.get("allPlayByPlay", [])
    for play in plays:
        # Append the result from each play if it exists
        result = play.get("result")
        if result:
            results_list.append(result)

# Count the occurrences of each result
result_counts = Counter(results_list)

# Print the result counts
print("Result Value Counts:")
for result, count in result_counts.items():
    print(f"{result}: {count}")


Result Value Counts:
Lineout: 38
Double: 28
Strikeout: 233
Single: 118
Walk: 71
Groundout: 187
Flyout: 102
Pop Out: 50
Double Play: 2
Sac Fly: 5
Home Run: 23
Hit By Pitch: 10
Forceout: 21
Intent Walk: 4
Grounded Into DP: 22
Field Error: 7
Triple: 2
Bunt Pop Out: 1
Caught Stealing 2B: 1
Bunt Groundout: 1
Fielders Choice Out: 1
Sac Bunt: 3
Fielders Choice: 1
Strikeout Double Play: 1
Caught Stealing Home: 1
